# Chapter 8 — Engine: PAST, PLFR, and Structured Prompt Frameworks

**Book:** Design of Agentic Systems with Case Studies  
**Type:** Type A — Architectural Pattern  
**Core Claim:** Structured prompt frameworks (PAST and PLFR) impose logical scaffolding that reduces ambiguity and improves output reliability at scale. Architecture — not the model — is the leverage point.

---

## What This Notebook Demonstrates

1. **The Ambiguity Multiplier** — Measuring how unstructured prompts degrade under task complexity.
2. **PAST Framework** — Linearizing execution with Problem → Action → Steps → Task.
3. **PLFR Framework** — Contracting reasoning with Prompt → Logic → Format → Result.
4. **The Failure Case** — Triggering the "confidently structured, silently wrong" failure where PAST without PLFR produces a correctly formatted but analytically wrong answer.
5. **Agentic Pipeline** — PAST as planning backbone + PLFR as agent-to-agent communication contracts.
6. **Human Decision Node** — Where the human overrides the AI's architectural proposal.

## Setup and Dependencies

In [1]:
# Install dependencies (uncomment if needed)
# !pip install openai pandas tabulate

import json
import os
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from enum import Enum
import textwrap

## Section 1: Synthetic Dataset — Churn by Plan Tier

We create a realistic churn dataset that exposes the failure mode. The key design:
- **Enterprise** has the highest absolute churn rate (stable across quarters).
- **Free** has the highest churn *increase* (Q3 spike).
- An unstructured or PAST-only prompt will identify Enterprise. The correct business answer is Free.

In [2]:
# Synthetic churn data — designed to expose the reasoning-method failure
churn_data = pd.DataFrame({
    'tier': ['Free', 'Pro', 'Enterprise'],
    'subscribers': [50000, 20000, 5000],
    'q1_churn_rate': [0.03, 0.05, 0.12],
    'q2_churn_rate': [0.035, 0.048, 0.115],
    'q3_churn_rate': [0.09, 0.055, 0.12],
})

# Derived columns
churn_data['q3_q2_delta'] = churn_data['q3_churn_rate'] - churn_data['q2_churn_rate']
churn_data['q3_q1_delta'] = churn_data['q3_churn_rate'] - churn_data['q1_churn_rate']
churn_data['weighted_contribution'] = (
    churn_data['q3_q2_delta'] * churn_data['subscribers'] / churn_data['subscribers'].sum()
)
churn_data['flag_5pct'] = churn_data['q3_churn_rate'] > 0.05

print("=== Churn Data by Plan Tier ===")
print(churn_data.to_string(index=False))
print(f"\nTier with highest absolute Q3 churn: {churn_data.loc[churn_data['q3_churn_rate'].idxmax(), 'tier']}")
print(f"Tier with highest Q3-Q2 delta:         {churn_data.loc[churn_data['q3_q2_delta'].idxmax(), 'tier']}")
print("\n⚠️  These are DIFFERENT answers. The prompt architecture determines which one the model returns.")

=== Churn Data by Plan Tier ===
      tier  subscribers  q1_churn_rate  q2_churn_rate  q3_churn_rate  q3_q2_delta  q3_q1_delta  weighted_contribution  flag_5pct
      Free        50000           0.03          0.035          0.090        0.055        0.060               0.036667       True
       Pro        20000           0.05          0.048          0.055        0.007        0.005               0.001867       True
Enterprise         5000           0.12          0.115          0.120        0.005        0.000               0.000333       True

Tier with highest absolute Q3 churn: Enterprise
Tier with highest Q3-Q2 delta:         Free

⚠️  These are DIFFERENT answers. The prompt architecture determines which one the model returns.


## Section 2: Framework Definitions

We define PAST and PLFR as structured Python dataclasses. These are not wrappers — they are the **architectural contracts** that govern how prompts are composed and validated.

In [3]:
@dataclass
class PASTPrompt:
    """PAST Framework: Linearizes task execution.

    Architectural Role:
    - Problem: scopes the reasoning space
    - Action: constrains the operation type
    - Steps: linearizes the execution plan
    - Task: anchors the output contract
    """
    problem: str
    action: str
    steps: List[str]
    task: str

    def render(self) -> str:
        steps_text = '\n'.join(f'  {i+1}. {s}' for i, s in enumerate(self.steps))
        return (
            f"Problem: {self.problem}\n\n"
            f"Action: {self.action}\n\n"
            f"Steps:\n{steps_text}\n\n"
            f"Task: {self.task}"
        )

    def validate(self) -> List[str]:
        """Structural validation — checks form, not reasoning."""
        issues = []
        if not self.problem:
            issues.append("Missing Problem: reasoning space is unbounded.")
        if not self.action:
            issues.append("Missing Action: operation type unconstrained.")
        if len(self.steps) < 2:
            issues.append("Fewer than 2 Steps: task may not need PAST decomposition.")
        if len(self.steps) > 7:
            issues.append("More than 7 Steps: consider splitting into sub-tasks.")
        if not self.task:
            issues.append("Missing Task: output contract undefined.")
        return issues


@dataclass
class PLFRPrompt:
    """PLFR Framework: Contracts the reasoning chain.

    Architectural Role:
    - Prompt: defines the reasoning entry point
    - Logic: constrains HOW the model thinks
    - Format: enforces output predictability
    - Result: establishes validation benchmark
    """
    prompt: str
    logic: str
    format_spec: str
    result: str

    def render(self) -> str:
        return (
            f"Prompt: {self.prompt}\n\n"
            f"Logic: {self.logic}\n\n"
            f"Format: {self.format_spec}\n\n"
            f"Result: {self.result}"
        )

    def validate(self) -> List[str]:
        """Structural validation — checks that reasoning is specified."""
        issues = []
        if not self.prompt:
            issues.append("Missing Prompt: no reasoning entry point.")
        if not self.logic:
            issues.append("⚠️  Missing Logic: reasoning method unconstrained. "
                         "This is the most common source of the 'structured but wrong' failure.")
        if not self.format_spec:
            issues.append("Missing Format: output shape undefined.")
        if not self.result:
            issues.append("Missing Result: no self-validation criteria.")
        return issues


print("✅ PAST and PLFR framework classes defined.")

✅ PAST and PLFR framework classes defined.


## Section 3: The Failure Case — PAST-Only vs. PAST+PLFR

This is the core demonstration. We construct two prompts for the same business question:
1. **PAST-only**: Correctly structured, but the Steps component says "identify the tier with the highest churn" (ambiguous — absolute or delta?).
2. **PAST+PLFR**: Same structure, but with a PLFR Logic component that locks the reasoning method to delta-based comparison.

We then simulate how each prompt directs an analytical function and observe the divergent results.

In [4]:
# === PAST-Only Prompt (THE DEFECTIVE VERSION) ===
past_only = PASTPrompt(
    problem="Our Q3 churn rate increased across multiple plan tiers. "
            "Leadership needs to know which tier is driving the increase.",
    action="Identify the primary churn driver by plan tier.",
    steps=[
        "Calculate Q3 churn rate for each plan tier (Free, Pro, Enterprise).",
        "Retrieve Q2 churn rates for the same tiers.",
        "Compare Q3 to Q2 for each tier.",           # ← AMBIGUOUS: compare how?
        "Identify the tier with the highest churn.",  # ← DEFECT: highest RATE, not highest INCREASE
        "Draft a one-sentence recommendation."
    ],
    task="Return the tier name, its churn rate, and a recommendation."
)

print("=== PAST-Only Prompt ===")
print(past_only.render())
print("\n--- Validation ---")
for issue in past_only.validate():
    print(f"  ⚠️  {issue}")
if not past_only.validate():
    print("  ✅ Structural validation passed. (But structure ≠ correctness.)")

=== PAST-Only Prompt ===
Problem: Our Q3 churn rate increased across multiple plan tiers. Leadership needs to know which tier is driving the increase.

Action: Identify the primary churn driver by plan tier.

Steps:
  1. Calculate Q3 churn rate for each plan tier (Free, Pro, Enterprise).
  2. Retrieve Q2 churn rates for the same tiers.
  3. Compare Q3 to Q2 for each tier.
  4. Identify the tier with the highest churn.
  5. Draft a one-sentence recommendation.

Task: Return the tier name, its churn rate, and a recommendation.

--- Validation ---
  ✅ Structural validation passed. (But structure ≠ correctness.)


In [5]:
# === PLFR Execution Contract (THE FIX) ===
plfr_logic = PLFRPrompt(
    prompt="Determine which plan tier is most responsible for the Q3 churn increase.",
    logic="Use delta-based comparison. For each tier, compute (Q3_churn_rate - Q2_churn_rate). "
          "The primary driver is the tier with the largest positive delta. "
          "Do NOT use absolute churn rate — a tier with stable high churn is not a 'driver' of increase.",
    format_spec='JSON: {"primary_driver": "<tier>", "delta": <float>, '
                '"breakdown": [{"tier": "<n>", "q2": <float>, "q3": <float>, "delta": <float>}]}',
    result="primary_driver should be the tier with the largest positive delta. "
           "If two tiers are within 0.5pp, flag as 'ambiguous'."
)

print("=== PLFR Execution Contract ===")
print(plfr_logic.render())
print("\n--- Validation ---")
for issue in plfr_logic.validate():
    print(f"  {issue}")
if not plfr_logic.validate():
    print("  ✅ All components present — reasoning method is specified.")

=== PLFR Execution Contract ===
Prompt: Determine which plan tier is most responsible for the Q3 churn increase.

Logic: Use delta-based comparison. For each tier, compute (Q3_churn_rate - Q2_churn_rate). The primary driver is the tier with the largest positive delta. Do NOT use absolute churn rate — a tier with stable high churn is not a 'driver' of increase.

Format: JSON: {"primary_driver": "<tier>", "delta": <float>, "breakdown": [{"tier": "<n>", "q2": <float>, "q3": <float>, "delta": <float>}]}

Result: primary_driver should be the tier with the largest positive delta. If two tiers are within 0.5pp, flag as 'ambiguous'.

--- Validation ---
  ✅ All components present — reasoning method is specified.


In [6]:
# === Simulated Execution: PAST-Only ===
# The model follows the steps literally. Step 4 says "highest churn" → absolute rate.

def execute_past_only(data: pd.DataFrame) -> dict:
    """Simulates model behavior under PAST-only prompt.
    Follows Step 4 literally: 'tier with the highest churn' → absolute Q3 rate.
    """
    idx = data['q3_churn_rate'].idxmax()
    row = data.iloc[idx]
    return {
        'method': 'PAST-only (absolute rate)',
        'primary_driver': row['tier'],
        'q3_churn_rate': row['q3_churn_rate'],
        'recommendation': f"Focus retention efforts on {row['tier']} tier "
                         f"(highest churn at {row['q3_churn_rate']:.1%})."
    }


# === Simulated Execution: PAST + PLFR ===
# The PLFR Logic component specifies delta-based comparison.

def execute_past_plfr(data: pd.DataFrame) -> dict:
    """Simulates model behavior under PAST+PLFR prompt.
    PLFR Logic: 'Use delta-based comparison' → Q3-Q2 delta.
    """
    idx = data['q3_q2_delta'].idxmax()
    row = data.iloc[idx]

    breakdown = []
    for _, r in data.iterrows():
        breakdown.append({
            'tier': r['tier'],
            'q2': r['q2_churn_rate'],
            'q3': r['q3_churn_rate'],
            'delta': r['q3_q2_delta']
        })

    # Check ambiguity per Result spec
    sorted_deltas = data.sort_values('q3_q2_delta', ascending=False)
    top_two_gap = sorted_deltas.iloc[0]['q3_q2_delta'] - sorted_deltas.iloc[1]['q3_q2_delta']

    return {
        'method': 'PAST+PLFR (delta-based)',
        'primary_driver': row['tier'],
        'delta': row['q3_q2_delta'],
        'ambiguous': top_two_gap < 0.005,
        'breakdown': breakdown
    }


# === Run both and compare ===
result_past = execute_past_only(churn_data)
result_plfr = execute_past_plfr(churn_data)

print("=" * 60)
print("FAILURE CASE DEMONSTRATION")
print("=" * 60)
print(f"\n📊 PAST-Only Result:")
print(f"   Primary Driver: {result_past['primary_driver']}")
print(f"   Metric Used:    Absolute Q3 churn rate ({result_past['q3_churn_rate']:.1%})")
print(f"   Recommendation: {result_past['recommendation']}")

print(f"\n📊 PAST+PLFR Result:")
print(f"   Primary Driver: {result_plfr['primary_driver']}")
print(f"   Metric Used:    Q3-Q2 delta ({result_plfr['delta']:.1%})")
print(f"   Ambiguous:      {result_plfr['ambiguous']}")
print(f"   Breakdown:")
for b in result_plfr['breakdown']:
    print(f"      {b['tier']:12s}  Q2={b['q2']:.1%}  Q3={b['q3']:.1%}  Δ={b['delta']:+.1%}")

print(f"\n🚨 DIVERGENT RESULTS:")
print(f"   PAST-only says: Focus on {result_past['primary_driver']} (stable high churn, no change)")
print(f"   PAST+PLFR says: Focus on {result_plfr['primary_driver']} (churn SPIKED this quarter)")
print(f"\n   The business decisions are OPPOSITE.")
print(f"   The model was correct both times — the architecture was different.")

FAILURE CASE DEMONSTRATION

📊 PAST-Only Result:
   Primary Driver: Enterprise
   Metric Used:    Absolute Q3 churn rate (12.0%)
   Recommendation: Focus retention efforts on Enterprise tier (highest churn at 12.0%).

📊 PAST+PLFR Result:
   Primary Driver: Free
   Metric Used:    Q3-Q2 delta (5.5%)
   Ambiguous:      False
   Breakdown:
      Free          Q2=3.5%  Q3=9.0%  Δ=+5.5%
      Pro           Q2=4.8%  Q3=5.5%  Δ=+0.7%
      Enterprise    Q2=11.5%  Q3=12.0%  Δ=+0.5%

🚨 DIVERGENT RESULTS:
   PAST-only says: Focus on Enterprise (stable high churn, no change)
   PAST+PLFR says: Focus on Free (churn SPIKED this quarter)

   The business decisions are OPPOSITE.
   The model was correct both times — the architecture was different.


## Section 4: AI Scaffold — Automated PAST Decomposition with Human Decision Node

This section implements an AI scaffold that:
1. Takes a natural language request and proposes a PAST decomposition.
2. Identifies potential parallelism in the steps.
3. **HALTS** at a Mandatory Human Decision Node before finalizing the pipeline topology.

The scaffold handles one bounded enumeration task (proposing steps and agent assignments) but **does not** finalize the architecture without human verification.

In [7]:
class AgentRole(Enum):
    DATA_AGENT = "data_agent"
    ANALYSIS_AGENT = "analysis_agent"
    WRITING_AGENT = "writing_agent"
    VALIDATION_AGENT = "validation_agent"


@dataclass
class PipelineStep:
    step_id: int
    description: str
    assigned_agent: AgentRole
    depends_on: List[int]
    plfr_contract: Optional[PLFRPrompt] = None


@dataclass
class PipelineProposal:
    past_decomposition: PASTPrompt
    steps: List[PipelineStep]
    proposed_topology: str  # 'sequential' or 'parallel' or 'mixed'
    parallelizable_groups: List[List[int]]  # groups of step_ids that COULD run in parallel

    def display(self):
        print("=" * 60)
        print("AI-PROPOSED PIPELINE")
        print("=" * 60)
        print(f"\nPAST Decomposition:")
        print(textwrap.indent(self.past_decomposition.render(), '  '))
        print(f"\nProposed Topology: {self.proposed_topology}")
        print(f"\nStep Assignments:")
        for s in self.steps:
            deps = f" (depends on: {s.depends_on})" if s.depends_on else " (no dependencies)"
            print(f"  Step {s.step_id}: [{s.assigned_agent.value}] {s.description}{deps}")
        if self.parallelizable_groups:
            print(f"\nParallelizable Groups: {self.parallelizable_groups}")


def ai_scaffold_decompose(user_request: str) -> PipelineProposal:
    """AI Scaffold: Proposes a PAST decomposition and agent assignments.

    This is a BOUNDED ENUMERATION task — the scaffold proposes,
    the human decides.
    """
    # Simulated AI decomposition for the churn analysis request
    past = PASTPrompt(
        problem="Q3 churn increased. Leadership needs a segmented analysis "
                "with root cause identification and recommendations.",
        action="Perform segmented churn analysis and generate recommendations.",
        steps=[
            "Extract Q1, Q2, Q3 churn rates by plan tier from the data warehouse.",
            "Compute quarter-over-quarter deltas for each tier.",
            "Cross-reference flagged tiers with NPS survey scores.",
            "Generate tier-specific recommendations with supporting data.",
        ],
        task="Produce a segmented churn report with a recommendations section."
    )

    steps = [
        PipelineStep(1, "Extract churn data from warehouse", AgentRole.DATA_AGENT, []),
        PipelineStep(2, "Compute QoQ deltas and flag tiers >5%", AgentRole.ANALYSIS_AGENT, [1]),
        PipelineStep(3, "Cross-reference flagged tiers with NPS scores", AgentRole.ANALYSIS_AGENT, [1]),
        PipelineStep(4, "Generate recommendations per flagged tier", AgentRole.WRITING_AGENT, [2, 3]),
    ]

    # AI PROPOSES: Steps 2 and 3 can run in parallel (both depend only on Step 1)
    return PipelineProposal(
        past_decomposition=past,
        steps=steps,
        proposed_topology='mixed',
        parallelizable_groups=[[2, 3]]  # AI thinks these can run together
    )


# Run the scaffold
proposal = ai_scaffold_decompose(
    "Analyze our Q3 churn by tier, cross-reference with NPS, and draft recommendations."
)
proposal.display()

AI-PROPOSED PIPELINE

PAST Decomposition:
  Problem: Q3 churn increased. Leadership needs a segmented analysis with root cause identification and recommendations.

  Action: Perform segmented churn analysis and generate recommendations.

  Steps:
    1. Extract Q1, Q2, Q3 churn rates by plan tier from the data warehouse.
    2. Compute quarter-over-quarter deltas for each tier.
    3. Cross-reference flagged tiers with NPS survey scores.
    4. Generate tier-specific recommendations with supporting data.

  Task: Produce a segmented churn report with a recommendations section.

Proposed Topology: mixed

Step Assignments:
  Step 1: [data_agent] Extract churn data from warehouse (no dependencies)
  Step 2: [analysis_agent] Compute QoQ deltas and flag tiers >5% (depends on: [1])
  Step 3: [analysis_agent] Cross-reference flagged tiers with NPS scores (depends on: [1])
  Step 4: [writing_agent] Generate recommendations per flagged tier (depends on: [2, 3])

Parallelizable Groups: [[2, 3]]


In [8]:
# ══════════════════════════════════════════════════════════════════
# ██  MANDATORY HUMAN DECISION NODE                              ██
# ══════════════════════════════════════════════════════════════════
#
# The AI scaffold proposed the following pipeline topology:
#
#   Step 1: Extract churn data            → data_agent
#   Step 2: Compute deltas                → analysis_agent  ┐
#   Step 3: Cross-reference NPS           → analysis_agent  ┤ PARALLEL?
#   Step 4: Generate recommendations      → writing_agent   (depends on 2,3)
#
# ARCHITECTURAL ASSUMPTION: Steps 2 and 3 can execute in parallel
# because they both depend only on Step 1 output.
#
# ⚠️  VERIFY THIS ASSUMPTION:
#   Does Step 3 (NPS cross-reference) need to know WHICH tiers
#   are flagged (>5% churn)? If yes, it depends on Step 2's output,
#   not just Step 1's. The correct topology is SEQUENTIAL.
#
#   The AI missed this dependency because it parsed the step
#   descriptions syntactically, not semantically. Step 3 says
#   "flagged tiers" — the AI did not trace where the flags originate.
#
# ═══════════════════════════════════════════════════════════════════

HUMAN_DECISION = "sequential"  # ← CHANGE THIS: "parallel" or "sequential"

HUMAN_REASONING = """
Step 3 says 'cross-reference FLAGGED tiers with NPS scores.' The flags
are computed in Step 2 (tiers where Q3 churn > 5%). Therefore Step 3
depends on Step 2's output — specifically, the list of flagged tier names.

If we run Steps 2 and 3 in parallel, Step 3 has no flag information and
must cross-reference ALL tiers with NPS, which:
  (a) wastes compute on tiers that aren't problematic,
  (b) produces a larger output that Step 4 must filter — pushing the
      filtering decision to the writing agent, which is not its job.

The AI proposed parallel execution because it matched dependency by
explicit Step IDs in the 'depends_on' field. It did not analyze the
SEMANTIC dependency between 'flagged tiers' in Step 3 and the flagging
operation in Step 2.

DECISION: Sequential. Step 3 depends on Step 2.
"""

print("HUMAN DECISION NODE")
print("=" * 40)
print(f"Decision:  {HUMAN_DECISION}")
print(f"Reasoning: {HUMAN_REASONING.strip()}")

HUMAN DECISION NODE
Decision:  sequential
Reasoning: Step 3 says 'cross-reference FLAGGED tiers with NPS scores.' The flags
are computed in Step 2 (tiers where Q3 churn > 5%). Therefore Step 3
depends on Step 2's output — specifically, the list of flagged tier names.

If we run Steps 2 and 3 in parallel, Step 3 has no flag information and
must cross-reference ALL tiers with NPS, which:
  (a) wastes compute on tiers that aren't problematic,
  (b) produces a larger output that Step 4 must filter — pushing the
      filtering decision to the writing agent, which is not its job.

The AI proposed parallel execution because it matched dependency by
explicit Step IDs in the 'depends_on' field. It did not analyze the
SEMANTIC dependency between 'flagged tiers' in Step 3 and the flagging
operation in Step 2.

DECISION: Sequential. Step 3 depends on Step 2.


In [9]:
# Apply the human decision to correct the pipeline

def apply_human_decision(proposal: PipelineProposal, decision: str) -> PipelineProposal:
    """Applies the human's topology decision to the pipeline."""
    if decision == "sequential":
        # Fix: Step 3 now depends on Step 2 (not just Step 1)
        corrected_steps = []
        for s in proposal.steps:
            if s.step_id == 3:
                corrected = PipelineStep(
                    step_id=3,
                    description=s.description,
                    assigned_agent=s.assigned_agent,
                    depends_on=[1, 2],  # ← CORRECTED: now depends on Step 2
                    plfr_contract=s.plfr_contract
                )
                corrected_steps.append(corrected)
            else:
                corrected_steps.append(s)

        return PipelineProposal(
            past_decomposition=proposal.past_decomposition,
            steps=corrected_steps,
            proposed_topology='sequential',
            parallelizable_groups=[]  # No parallel groups after correction
        )
    else:
        return proposal  # Accept AI proposal as-is


corrected_pipeline = apply_human_decision(proposal, HUMAN_DECISION)

print("=== CORRECTED PIPELINE (After Human Decision) ===")
corrected_pipeline.display()

print("\n✅ Pipeline topology corrected: Step 3 now depends on Step 2.")
print("   The AI's syntactic dependency analysis was overridden by")
print("   the human's semantic analysis of data flow.")

=== CORRECTED PIPELINE (After Human Decision) ===
AI-PROPOSED PIPELINE

PAST Decomposition:
  Problem: Q3 churn increased. Leadership needs a segmented analysis with root cause identification and recommendations.

  Action: Perform segmented churn analysis and generate recommendations.

  Steps:
    1. Extract Q1, Q2, Q3 churn rates by plan tier from the data warehouse.
    2. Compute quarter-over-quarter deltas for each tier.
    3. Cross-reference flagged tiers with NPS survey scores.
    4. Generate tier-specific recommendations with supporting data.

  Task: Produce a segmented churn report with a recommendations section.

Proposed Topology: sequential

Step Assignments:
  Step 1: [data_agent] Extract churn data from warehouse (no dependencies)
  Step 2: [analysis_agent] Compute QoQ deltas and flag tiers >5% (depends on: [1])
  Step 3: [analysis_agent] Cross-reference flagged tiers with NPS scores (depends on: [1, 2])
  Step 4: [writing_agent] Generate recommendations per flagged t

## Section 5: PLFR Execution Contracts for Each Agent

Now we attach PLFR contracts to each pipeline step. This is the **agent-to-agent communication layer** — each downstream agent receives a structured instruction that specifies reasoning method, output format, and validation criteria.

In [10]:
# Attach PLFR contracts to each step

plfr_contracts = {
    1: PLFRPrompt(
        prompt="Extract Q1, Q2, Q3 churn rates for each plan tier.",
        logic="Query the data warehouse table 'churn_metrics' with GROUP BY tier, quarter. "
              "Return rates as decimals (0.05, not 5%).",
        format_spec='{"tiers": [{"name": str, "q1": float, "q2": float, "q3": float}]}',
        result="Exactly 3 tiers. All rates between 0.0 and 1.0. No nulls."
    ),
    2: PLFRPrompt(
        prompt="Compute QoQ churn deltas and flag tiers exceeding 5% Q3 churn.",
        logic="Delta = Q3_rate - Q2_rate. Flag if Q3_rate > 0.05. "
              "Primary driver = tier with largest POSITIVE delta (not highest absolute rate).",
        format_spec='{"deltas": [{"tier": str, "delta": float, "flagged": bool}], '
                    '"primary_driver": str}',
        result="primary_driver is the tier with max delta. Deltas sum should approximate "
               "the overall churn change when weighted by subscriber share."
    ),
    3: PLFRPrompt(
        prompt="Cross-reference FLAGGED tiers (from Step 2) with NPS survey data.",
        logic="For each flagged tier ONLY, retrieve the most recent NPS score. "
              "Compute NPS-churn correlation direction (positive/negative/none). "
              "Do NOT analyze unflagged tiers.",
        format_spec='{"cross_ref": [{"tier": str, "nps_score": float, '
                    '"correlation_direction": str}]}',
        result="Only flagged tiers should appear. correlation_direction must be one of: "
               "'negative' (low NPS, high churn), 'positive' (unexpected), 'none'."
    ),
    4: PLFRPrompt(
        prompt="Generate one actionable recommendation per flagged tier.",
        logic="Each recommendation must reference: (a) the churn delta from Step 2, "
              "(b) the NPS correlation from Step 3. Recommendation must be a specific action, "
              "not a vague suggestion like 'improve retention'.",
        format_spec='{"recommendations": [{"tier": str, "action": str, '
                    '"supporting_data": str}]}',
        result="Each recommendation cites specific numbers. No recommendation should "
               "apply generically to all tiers."
    )
}

print("=== PLFR Execution Contracts ===")
for step_id, contract in plfr_contracts.items():
    print(f"\n--- Step {step_id} ---")
    issues = contract.validate()
    if issues:
        for issue in issues:
            print(f"  ⚠️  {issue}")
    else:
        print(f"  ✅ Complete PLFR contract. Logic: {contract.logic[:60]}...")

=== PLFR Execution Contracts ===

--- Step 1 ---
  ✅ Complete PLFR contract. Logic: Query the data warehouse table 'churn_metrics' with GROUP BY...

--- Step 2 ---
  ✅ Complete PLFR contract. Logic: Delta = Q3_rate - Q2_rate. Flag if Q3_rate > 0.05. Primary d...

--- Step 3 ---
  ✅ Complete PLFR contract. Logic: For each flagged tier ONLY, retrieve the most recent NPS sco...

--- Step 4 ---
  ✅ Complete PLFR contract. Logic: Each recommendation must reference: (a) the churn delta from...


## Section 6: Triggering the Failure — Remove the Logic Component

### Exercise for the Reader

The cell below creates a **defective** version of the Step 2 PLFR contract with the Logic component removed. This simulates what happens in production when teams use PAST for task structure but skip PLFR's reasoning specification.

**Prediction:** Without the Logic component, the analysis agent will default to absolute rate comparison (the most common interpretation of "highest churn") instead of delta-based comparison.

In [11]:
# === DEFECTIVE CONTRACT: Logic component removed ===
defective_contract = PLFRPrompt(
    prompt="Compute QoQ churn deltas and flag tiers exceeding 5% Q3 churn.",
    logic="",  # ← DELIBERATELY EMPTY: This is the failure trigger
    format_spec='{"deltas": [{"tier": str, "delta": float, "flagged": bool}], '
                '"primary_driver": str}',
    result="primary_driver is the tier with max delta."
)

print("=== Defective PLFR Contract (No Logic) ===")
print(defective_contract.render())
print("\n--- Validation ---")
for issue in defective_contract.validate():
    print(f"  {issue}")

=== Defective PLFR Contract (No Logic) ===
Prompt: Compute QoQ churn deltas and flag tiers exceeding 5% Q3 churn.

Logic: 

Format: {"deltas": [{"tier": str, "delta": float, "flagged": bool}], "primary_driver": str}

Result: primary_driver is the tier with max delta.

--- Validation ---
  ⚠️  Missing Logic: reasoning method unconstrained. This is the most common source of the 'structured but wrong' failure.


In [12]:
import random

def simulate_nondeterministic_execution(data: pd.DataFrame, n_runs: int = 5) -> None:
    """Simulates multiple LLM runs without a Logic component.

    Without Logic, the model non-deterministically chooses between:
    - Absolute rate comparison
    - Delta comparison
    - Percentage change comparison

    This demonstrates the 'structured but inconsistent' failure.
    """
    methods = [
        ('absolute_rate', lambda d: d.loc[d['q3_churn_rate'].idxmax(), 'tier']),
        ('delta', lambda d: d.loc[d['q3_q2_delta'].idxmax(), 'tier']),
        ('pct_change', lambda d: d.loc[
            ((d['q3_churn_rate'] - d['q2_churn_rate']) / d['q2_churn_rate']).idxmax(), 'tier'
        ]),
    ]

    print("=== Non-Deterministic Execution (No Logic Component) ===")
    print(f"Running {n_runs} simulated LLM calls...\n")

    results = []
    for i in range(n_runs):
        method_name, method_fn = random.choice(methods)
        result = method_fn(data)
        results.append((method_name, result))
        print(f"  Run {i+1}: Method={method_name:15s} → Primary Driver = {result}")

    unique_answers = set(r[1] for r in results)
    print(f"\n  Unique answers across {n_runs} runs: {unique_answers}")
    if len(unique_answers) > 1:
        print("  🚨 INCONSISTENT: Same prompt, same data, different answers.")
        print("     This is the 'structured but wrong' failure in action.")
    else:
        print("  (Run more times to see inconsistency — try n_runs=20)")


random.seed(None)  # Ensure non-determinism
simulate_nondeterministic_execution(churn_data, n_runs=10)

=== Non-Deterministic Execution (No Logic Component) ===
Running 10 simulated LLM calls...

  Run 1: Method=pct_change      → Primary Driver = Free
  Run 2: Method=pct_change      → Primary Driver = Free
  Run 3: Method=absolute_rate   → Primary Driver = Enterprise
  Run 4: Method=pct_change      → Primary Driver = Free
  Run 5: Method=absolute_rate   → Primary Driver = Enterprise
  Run 6: Method=absolute_rate   → Primary Driver = Enterprise
  Run 7: Method=pct_change      → Primary Driver = Free
  Run 8: Method=delta           → Primary Driver = Free
  Run 9: Method=absolute_rate   → Primary Driver = Enterprise
  Run 10: Method=delta           → Primary Driver = Free

  Unique answers across 10 runs: {'Free', 'Enterprise'}
  🚨 INCONSISTENT: Same prompt, same data, different answers.
     This is the 'structured but wrong' failure in action.


## Section 7: Full Pipeline Execution (Corrected)

This section runs the complete PAST+PLFR pipeline with the corrected sequential topology, demonstrating the end-to-end flow from decomposition through agent execution to final output.

In [13]:
# Simulated NPS data
nps_data = {
    'Free': 32,
    'Pro': 58,
    'Enterprise': 45
}


def run_full_pipeline(data: pd.DataFrame, nps: dict, contracts: dict) -> dict:
    """Executes the full corrected pipeline: PAST decomposition + PLFR execution."""

    print("=" * 60)
    print("FULL PIPELINE EXECUTION (PAST + PLFR, Sequential)")
    print("=" * 60)

    # Step 1: Data extraction
    print("\n▶ Step 1: [data_agent] Extracting churn data...")
    step1_output = {
        'tiers': [
            {'name': r['tier'], 'q1': r['q1_churn_rate'],
             'q2': r['q2_churn_rate'], 'q3': r['q3_churn_rate']}
            for _, r in data.iterrows()
        ]
    }
    print(f"  Output: {json.dumps(step1_output, indent=2)[:200]}...")

    # Step 2: Delta computation (using PLFR Logic: delta-based)
    print("\n▶ Step 2: [analysis_agent] Computing deltas (PLFR: delta-based)...")
    step2_output = {
        'deltas': [],
        'primary_driver': None
    }
    max_delta = -999
    for t in step1_output['tiers']:
        delta = t['q3'] - t['q2']
        flagged = t['q3'] > 0.05
        step2_output['deltas'].append({
            'tier': t['name'], 'delta': round(delta, 4), 'flagged': flagged
        })
        if delta > max_delta:
            max_delta = delta
            step2_output['primary_driver'] = t['name']

    print(f"  Output: {json.dumps(step2_output, indent=2)}")

    # Step 3: NPS cross-reference (SEQUENTIAL — uses Step 2 flags)
    print("\n▶ Step 3: [analysis_agent] Cross-referencing NPS (flagged tiers only)...")
    flagged_tiers = [d['tier'] for d in step2_output['deltas'] if d['flagged']]
    step3_output = {'cross_ref': []}
    for tier in flagged_tiers:
        nps_score = nps.get(tier, None)
        # Simple correlation: NPS < 40 with high churn = negative correlation
        if nps_score and nps_score < 40:
            corr = 'negative'
        elif nps_score and nps_score > 55:
            corr = 'none'  # High NPS but still churning — unexpected
        else:
            corr = 'none'
        step3_output['cross_ref'].append({
            'tier': tier, 'nps_score': nps_score, 'correlation_direction': corr
        })

    print(f"  Output: {json.dumps(step3_output, indent=2)}")

    # Step 4: Recommendations
    print("\n▶ Step 4: [writing_agent] Generating recommendations...")
    step4_output = {'recommendations': []}
    for ref in step3_output['cross_ref']:
        tier = ref['tier']
        delta_info = next(d for d in step2_output['deltas'] if d['tier'] == tier)

        if ref['correlation_direction'] == 'negative':
            action = (f"Launch targeted NPS recovery campaign for {tier} tier. "
                     f"NPS of {ref['nps_score']} correlates with churn spike of "
                     f"{delta_info['delta']:+.1%}. Prioritize support ticket "
                     f"response time and onboarding flow improvements.")
        else:
            action = (f"Investigate {tier} tier churn drivers beyond satisfaction — "
                     f"NPS of {ref['nps_score']} does not explain the "
                     f"{delta_info['delta']:+.1%} churn increase. Check pricing, "
                     f"competitive alternatives, and feature usage patterns.")

        step4_output['recommendations'].append({
            'tier': tier,
            'action': action,
            'supporting_data': f"Q3-Q2 Δ={delta_info['delta']:+.1%}, NPS={ref['nps_score']}"
        })

    print(f"  Output: {json.dumps(step4_output, indent=2)}")

    return {
        'step1': step1_output,
        'step2': step2_output,
        'step3': step3_output,
        'step4': step4_output
    }


pipeline_output = run_full_pipeline(churn_data, nps_data, plfr_contracts)

FULL PIPELINE EXECUTION (PAST + PLFR, Sequential)

▶ Step 1: [data_agent] Extracting churn data...
  Output: {
  "tiers": [
    {
      "name": "Free",
      "q1": 0.03,
      "q2": 0.035,
      "q3": 0.09
    },
    {
      "name": "Pro",
      "q1": 0.05,
      "q2": 0.048,
      "q3": 0.055
    },
    {
 ...

▶ Step 2: [analysis_agent] Computing deltas (PLFR: delta-based)...
  Output: {
  "deltas": [
    {
      "tier": "Free",
      "delta": 0.055,
      "flagged": true
    },
    {
      "tier": "Pro",
      "delta": 0.007,
      "flagged": true
    },
    {
      "tier": "Enterprise",
      "delta": 0.005,
      "flagged": true
    }
  ],
  "primary_driver": "Free"
}

▶ Step 3: [analysis_agent] Cross-referencing NPS (flagged tiers only)...
  Output: {
  "cross_ref": [
    {
      "tier": "Free",
      "nps_score": 32,
      "correlation_direction": "negative"
    },
    {
      "tier": "Pro",
      "nps_score": 58,
      "correlation_direction": "none"
    },
    {
      "tier": 

## Section 8: Comparison Summary

Final side-by-side comparison of all three approaches.

In [14]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║                    ARCHITECTURAL COMPARISON                        ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  Approach 1: Unstructured Prompt                                   ║
║  ─────────────────────────────────                                 ║
║  "Analyze Q3 churn by tier and recommend actions."                 ║
║  → Non-deterministic method, format, and scope                     ║
║  → Cannot reproduce results across runs                            ║
║  → Ambiguity: O(n^k) where k = unstated dependencies              ║
║                                                                    ║
║  Approach 2: PAST-Only                                             ║
║  ──────────────────────                                            ║
║  Structured steps, defined output format.                          ║
║  → Deterministic sequence and format                               ║
║  → NON-deterministic reasoning method                              ║
║  → Result: Enterprise (highest absolute rate = WRONG ANSWER)       ║
║  → ⚠️  Looks correct. Passes format validation. Fails business.   ║
║                                                                    ║
║  Approach 3: PAST + PLFR                                           ║
║  ────────────────────────                                          ║
║  Structured steps + specified reasoning method.                    ║
║  → Deterministic sequence, format, AND method                      ║
║  → Result: Free (highest delta = CORRECT ANSWER)                   ║
║  → Reproducible and auditable                                      ║
║                                                                    ║
║  The model didn't change. The architecture did.                    ║
╚══════════════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════════════╗
║                    ARCHITECTURAL COMPARISON                        ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  Approach 1: Unstructured Prompt                                   ║
║  ─────────────────────────────────                                 ║
║  "Analyze Q3 churn by tier and recommend actions."                 ║
║  → Non-deterministic method, format, and scope                     ║
║  → Cannot reproduce results across runs                            ║
║  → Ambiguity: O(n^k) where k = unstated dependencies              ║
║                                                                    ║
║  Approach 2: PAST-Only                                             ║
║  ──────────────────────                                            ║
║  Structured steps, defined output format.                          ║
║ 

## Appendix: Framework Quick Reference

| Framework | Components | Architectural Role | Best For | Failure Without It |
|-----------|------------|-------------------|----------|-----------------|
| **PAST** | Problem, Action, Steps, Task | Linearizes execution | Multi-step procedural tasks | Step omission, scope ambiguity |
| **PLFR** | Prompt, Logic, Format, Result | Contracts reasoning | Analytical/evaluative tasks | Wrong reasoning method, inconsistent results |
| **PAST+PLFR** | Combined | Full pipeline control | Agentic systems | The one you should use |

---

*This notebook accompanies Chapter 8 of Design of Agentic Systems with Case Studies.*